# Enhanced Deltashare iSpot Report Optimization

## Overview

This notebook compares two view definitions used to produce the enhanced commercial report for iSpot, both querying `prod.detection.viewing_commercials_dedup_golden` joined to `prod.customer_reports.hashes_production_vizio`.

| View | Cell | Purpose |
| --- | --- | --- |
| `dev.mohit_gangwani.test_enhanced_ispot_existing` | Cell 9 | Existing template-style definition |
| `dev.mohit_gangwani.test_enhanced_ispot_refined` | Cell 10 | Refined definition with cleaner query shape |

The refined view keeps the same output shape and returned identical benchmark results, but changes the query shape in three ways:

* removes dead-code CASE branches that Catalyst constant-folds away at parse time
* precomputes `prev_filter` and `next_filter` in the source subquery so the blacklist condition is evaluated once per row instead of once per projected column
* moves `commercial_client` and `fk_commercial_source_id` partition filters into the subquery and simplifies the `acrb_clients` filter to a direct boolean expression

After the benchmark runs, Cell 10 was further updated to remove `SELECT DISTINCT` (confirmed redundant by the query profile — zero rows were deduplicated on this pre-deduplicated table) and the `/*+ BROADCAST(v) */` hint (silently ignored because `hashes_production_vizio` is ~190M rows, far above the broadcast threshold).

The goal of the notebook is to show both the logical-plan differences and the measured runtime impact of those changes on serverless compute.

## Problem 1: Dead-Code CASE Branches Across Nine Projected Columns

The existing view contains six CASE branches per output column, of which several Catalyst constant-folds to always-true or always-false because the template used string literals from a parameterized client configuration:

```sql
CASE WHEN False THEN NULL                                        -- always false, never reached
     WHEN (excluded_client_list IS NOT NULL
           AND excluded_client_list != '||'
           AND excluded_client_list NOT LIKE '%|ispot|%')
           OR excluded_client_list <=> '|ALL|' THEN NULL        -- the real blacklist check
     WHEN 'ispot' != 'nielsen' AND prev_nielsen_exclusive THEN NULL  -- 'ispot' != 'nielsen' is always True
     WHEN False = False AND prev_station_vod = True THEN NULL        -- False = False is always True
     WHEN 'TMS' = 'TMS' THEN mt.tms_prev_episode_id                  -- always True, absorbs everything below
     WHEN 'TMS' = 'TIVO' THEN mt.tivo_prev_episode_id               -- always False, never reached
END
```

This same six-branch pattern repeats across all nine `prev_*` / `next_*` projected columns.

Why this matters:

* Catalyst does constant-fold the dead branches in the optimized plan, so `WHEN False` and `WHEN 'TMS' = 'TIVO'` are removed at compile time
* However `'ispot' != 'nielsen'` and `False = False` are folded to `true` but remain as WHEN branches in the source, making the view fragile — adding another constant-compare branch in the wrong position could silently mask the blacklist check
* The expression tree presented to the optimizer is wider than necessary, and the intent is completely obscured by the leftover template scaffolding

## Fix 1: Remove Dead-Code Branches

Removing the dead branches leaves only the three real conditions per column, which are then further collapsed into the `prev_filter` / `next_filter` precomputation described in Fix 2.

In the existing plan (Cell 12), Catalyst does constant-fold `WHEN False` and `WHEN 'TMS' = 'TIVO'` away. The folded `WHEN 'TMS' = 'TMS'` becomes `WHEN true`, which means the plan retains that branch as an unconditional value pick. The final constant-folded shape in the Cell 12 plan for each prev column is:

```sql
CASE WHEN (excluded_client_list <=> '|ALL|'
           OR (isnotnull(excluded_client_list)
               AND NOT (excluded_client_list = '||')
               AND NOT Contains(excluded_client_list, '|ispot|'))) THEN null
     WHEN prev_nielsen_exclusive THEN null
     WHEN prev_station_vod THEN null
     WHEN true THEN tms_prev_episode_id  -- folded from 'TMS' = 'TMS'
END
```

In the refined view the intent is explicit with no constant-folded scaffolding. The active conditions are precomputed as `prev_filter` and `next_filter` so each projected column becomes two branches:

```sql
CASE WHEN mt.prev_filter THEN NULL
     ELSE mt.tms_prev_episode_id
END
```

The EXPLAIN COST for Cell 13 confirms this: Catalyst shows `CASE WHEN prev_filter#116041 THEN null ELSE tms_prev_episode_id#116186 END` in every prev column, with a single attribute reference rather than an inline expression.

## Problem 2: `excluded_client_list` Blacklist Condition Repeated Across Nine Projected Columns

After Catalyst constant-folds the dead branches, the blacklist condition that actually matters is:

```sql
(excluded_client_list IS NOT NULL
 AND excluded_client_list != '||'
 AND excluded_client_list NOT LIKE '%|ispot|%')
OR excluded_client_list <=> '|ALL|'
OR prev_nielsen_exclusive
OR prev_station_vod
```

This four-part expression is inlined separately inside each of the nine CASE blocks — five `prev_*` columns using `prev_nielsen_exclusive` / `prev_station_vod`, and four `next_*` columns using `next_nielsen_exclusive` / `next_station_vod`.

Why this hurts performance:

* Photon evaluates the full four-condition expression fresh per row per column — the `NOT LIKE '%|ispot|%'` pattern match runs nine times per row instead of once
* the EXPLAIN COST plan for Cell 12 shows the full `excluded_client_list` expression inlined in every CASE column rather than shared as a named attribute
* the wider expression tree increases per-row projection CPU cost across all output rows

## Fix 2: Precompute `prev_filter` and `next_filter` in the Source Subquery

The refined view adds two boolean columns directly to the source subquery, evaluated once per row:

```sql
FROM (
    SELECT viz.*,
        (COALESCE(viz.excluded_client_list, '||') != '||'
         AND viz.excluded_client_list NOT LIKE '%|ispot|%')
        OR viz.excluded_client_list <=> '|ALL|'
        OR viz.prev_nielsen_exclusive
        OR viz.prev_station_vod AS prev_filter,

        (COALESCE(viz.excluded_client_list, '||') != '||'
         AND viz.excluded_client_list NOT LIKE '%|ispot|%')
        OR viz.excluded_client_list <=> '|ALL|'
        OR viz.next_nielsen_exclusive
        OR viz.next_station_vod AS next_filter
    FROM prod.detection.viewing_commercials_dedup_golden AS viz
    WHERE ...
) AS mt
```

Each outer CASE expression then becomes a simple two-branch lookup against a boolean column:

```sql
CASE WHEN mt.prev_filter THEN NULL ELSE mt.tms_prev_episode_id END
```

The EXPLAIN COST for Cell 13 confirms this: the optimized plan shows `CASE WHEN prev_filter#116041 THEN null ELSE ...` and `CASE WHEN next_filter#116042 THEN null ELSE ...` as single attribute references in all nine projected columns, rather than expanding the four-condition expression per column as in the Cell 12 plan.

## Problem 3: WHERE Clause Complexity and Partition Filter Placement

The existing view has three WHERE-related issues that add optimizer overhead even though Catalyst partially mitigates them at runtime.

**`acrb_clients` filter uses a CASE anti-pattern:**

```sql
WHERE CASE WHEN (viz.acrb_clients IS NOT NULL
      AND viz.acrb_clients != '||'
      AND viz.acrb_clients NOT LIKE '%|ispot|%') OR viz.acrb_clients <=> '|ALL|'
      THEN FALSE ELSE TRUE END
```

`CASE WHEN condition THEN FALSE ELSE TRUE` is equivalent to `NOT condition` — the CASE form prevents the optimizer from recognizing it as a simple boolean filter.

**Partition filters are in the outer WHERE, not the subquery:**

```sql
WHERE ...
AND mt.commercial_client = 'ispot'
AND fk_commercial_source_id IN (1)
```

Catalyst does push these down to the scan in practice (both profiles show 1 partition read), but placing them in the outer WHERE creates an indirect path for predicate pushdown and is fragile if the subquery shape changes.

**Outer WHERE has a dead third branch:**

```sql
WHERE (mt.prev_vizio_epg_not_null != False
      OR NOT(mt.acrb_clients <=> '|ALL|' AND '' <=> mt.app_service)
      OR (NOT(mt.acrb_clients IS NOT NULL ...) AND NOT(mt.acrb_clients <=> '|ALL|'))
)
```

The third condition requires `acrb_clients IS DISTINCT FROM '|ALL|'` — but the subquery WHERE already removes all rows where `acrb_clients <=> '|ALL|'`. After the subquery filter, the second condition's `acrb_clients IS DISTINCT FROM '|ALL|'` is always true, making the third branch unreachable. The entire outer CASE becomes a no-op after the subquery filter.

## Fix 3: Simplified WHERE and Partition Filters Moved to Subquery

The refined view makes three related WHERE changes.

**`acrb_clients` filter rewritten as a direct boolean:**

```sql
-- Existing (CASE anti-pattern)
WHERE CASE WHEN (viz.acrb_clients IS NOT NULL AND viz.acrb_clients != '||'
           AND viz.acrb_clients NOT LIKE '%|ispot|%') OR viz.acrb_clients <=> '|ALL|'
      THEN FALSE ELSE TRUE END

-- Refined (direct NOT)
WHERE NOT ((COALESCE(viz.acrb_clients, '||') != '||'
            AND viz.acrb_clients NOT LIKE '%|ispot|%')
           OR viz.acrb_clients <=> '|ALL|')
```

The refined form is a transparent boolean filter that Catalyst can push directly without CASE-expression reasoning.

**Partition filters moved into the subquery WHERE:**

```sql
WHERE NOT (...)           -- acrb_clients check
  AND viz.fk_commercial_source_id = 1
  AND viz.commercial_client = 'ispot'
```

This makes the partition pruning explicit and co-located with the scan. The EXPLAIN plan for Cell 13 confirms both filters appear directly on the `Relation prod.detection.viewing_commercials_dedup_golden` node.

**Outer WHERE removed entirely:**

Because the subquery WHERE already enforces `commercial_client = 'ispot'`, `fk_commercial_source_id = 1`, and the `acrb_clients` access control, the outer CASE block becomes unconditionally true for all rows the subquery returns. The refined view has no outer WHERE clause — the JOIN condition is the only downstream filter needed.

## Measured Performance Results

Benchmark query profiles:

* existing view (Cell 15, ran first): `0cd5355e-7a62-4ba1-973e-99dd337b1c9b`
* refined view (Cell 16, ran second): `09a6539f-72f9-4a18-95ce-e88b8b7a0974`

> **Run-order note:** The existing benchmark ran first (0% cache hit — all remote reads). The refined benchmark ran second (37% cache hit). The performance gap is primarily structural — task time dropped 26% and wall-clock by 75%, with the Shuffle operator (the dominant cost) cut nearly in half.

| Metric | Existing | Refined | Notes |
| --- | --- | --- | --- |
| `COUNT(1)` | 1,309,167,248 | 1,309,167,248 | identical |
| `COUNT(DISTINCT tvid)` | 10,028,320 | 10,028,320 | identical |
| `COUNT(DISTINCT value)` | 231,763 | 231,763 | identical |
| Duration | 371.2s (6m 11s) | **93.7s (1m 33.7s)** | **−74.8% (3.96× faster)** |
| Read bytes | ~74.3 GiB | ~74.5 GiB | ~same — same ispot partition |
| Remote bytes | ~74.1 GiB | ~46.4 GiB | existing ran cold |
| Cache hit | 0% | 37% | existing ran first |
| Task time | 4.98h | 3.68h | **−26.1%** |
| Shuffle | 3.44h | 1.78h | **−48.3%** — dominant driver |
| Grouping Aggregate | 42.57m | 1.12h | existing: shuffle-dominated; refined: balanced |
| Files read | 124 | 124 | same — single ispot partition |
| Partitions read | 1 | 1 | same pruning |
| Pruned bytes | 251.0 TiB | 251.0 TiB | same pruning |

**Interpretation:**

The 74.8% wall-clock improvement is driven by the Shuffle cost reduction: 3.44h → 1.78h (−48.3%). The existing view’s complex inline CASE expressions — a 4-condition `excluded_client_list` check repeated across 9 columns — require significantly more data to evaluate and shuffle before the dedup aggregate than the refined view’s single-attribute `prev_filter` / `next_filter` lookups. The direct partition filter pushdown in the refined subquery also ensures only ispot rows enter the pipeline rather than being filtered at the outer WHERE after a broader read.

Both views produce identical results: existing uses `SELECT DISTINCT`, refined uses `GROUP BY ALL` (equivalent semantics). Since the source is a single pre-deduplicated table (`viewing_commercials_dedup_golden`), both dedup operations confirm zero effective row removal — the dedup overhead is the residual cost of the view design.

In [0]:
%sql
CREATE OR REPLACE VIEW dev.mohit_gangwani.test_enhanced_ispot_existing AS
SELECT /*+ BROADCAST(tp)
      BROADCAST(pop)
         */
DISTINCT
mt.tvid AS tvid,
mt.zipcode as zipcode, 
mt.dma as dma, 
mt.external_id as value, 
mt.mt_start as mt_start, 
mt.session_start as ts_start, 
mt.session_end as ts_end, 
CASE WHEN False THEN NULL 
WHEN (mt.excluded_client_list IS NOT NULL 
      AND mt.excluded_client_list != '||' 
      AND mt.excluded_client_list NOT LIKE '%|ispot|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
WHEN 'ispot' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
WHEN False = False AND mt.prev_station_vod = True THEN NULL 
WHEN 'TMS' = 'TMS'  THEN mt.tms_prev_episode_id 
WHEN 'TMS' = 'TIVO' THEN mt.tivo_prev_episode_id 
END as prev_episode_id, 
CASE 
WHEN False THEN NULL 
      WHEN (mt.excluded_client_list IS NOT NULL 
      AND mt.excluded_client_list != '||' 
      AND mt.excluded_client_list NOT LIKE '%|ispot|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
WHEN 'ispot' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
WHEN False = False AND mt.prev_station_vod = True THEN NULL 
WHEN 'ispot' = 'nielsen'   THEN mt.tms_prev_show_title 
WHEN 'TMS' = 'TMS'  THEN COALESCE(mt.tms_prev_show_title, mt.tivo_prev_show_title) 
WHEN 'TMS' = 'TIVO' THEN COALESCE(mt.tivo_prev_show_title, mt.tms_prev_show_title) 
END as prev_title, 
mt.prev_ts_start as prev_ts_start, 
mt.prev_ts_end as prev_ts_end, 
CASE WHEN False THEN NULL 
      WHEN (mt.excluded_client_list IS NOT NULL 
      AND mt.excluded_client_list != '||' 
      AND mt.excluded_client_list NOT LIKE '%|ispot|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
WHEN 'ispot' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
WHEN False = False AND mt.prev_station_vod = True THEN NULL 
WHEN 'ispot' = 'nielsen'   THEN mt.tms_prev_channel_callsign 
WHEN 'TMS' = 'TMS'  THEN COALESCE(mt.tms_prev_channel_callsign, mt.tivo_prev_channel_callsign) 
WHEN 'TMS' = 'TIVO' THEN COALESCE(mt.tivo_prev_channel_callsign, mt.tms_prev_channel_callsign) 
END as prev_channel_callsign, 
CASE WHEN False THEN NULL 
      WHEN (mt.excluded_client_list IS NOT NULL 
      AND mt.excluded_client_list != '||' 
      AND mt.excluded_client_list NOT LIKE '%|ispot|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
WHEN 'ispot' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
WHEN False = False AND mt.prev_station_vod = True THEN NULL 
WHEN 'ispot' = 'nielsen'   THEN mt.tms_prev_network_affiliate 
WHEN 'TMS' = 'TMS'  THEN COALESCE(mt.tms_prev_network_affiliate, mt.tivo_prev_network_affiliate) 
WHEN 'TMS' = 'TIVO' THEN COALESCE(mt.tivo_prev_network_affiliate, mt.tms_prev_network_affiliate) 
END as prev_network_affiliate, 
CASE WHEN False THEN NULL 
WHEN (mt.excluded_client_list IS NOT NULL 
      AND mt.excluded_client_list != '||' 
      AND mt.excluded_client_list NOT LIKE '%|ispot|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
WHEN 'ispot' != 'nielsen' AND mt.next_nielsen_exclusive THEN NULL 
WHEN False = False AND mt.next_station_vod = True THEN NULL 
WHEN 'TMS' = 'TMS'  THEN mt.tms_next_episode_id 
WHEN 'TMS' = 'TIVO' THEN mt.tivo_next_episode_id 
END as next_episode_id, 
CASE 
WHEN False THEN NULL 
      WHEN (mt.excluded_client_list IS NOT NULL 
      AND mt.excluded_client_list != '||' 
      AND mt.excluded_client_list NOT LIKE '%|ispot|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
WHEN 'ispot' != 'nielsen' AND mt.next_nielsen_exclusive THEN NULL 
WHEN False = False AND mt.next_station_vod = True THEN NULL 
WHEN 'ispot' = 'nielsen'   THEN mt.tms_next_show_title 
WHEN 'TMS' = 'TMS'  THEN COALESCE(mt.tms_next_show_title, mt.tivo_next_show_title) 
WHEN 'TMS' = 'TIVO' THEN COALESCE(mt.tivo_next_show_title, mt.tms_next_show_title) 
END as next_title, 
mt.next_ts_start as next_ts_start, 
mt.next_ts_end as next_ts_end, 
CASE WHEN False THEN NULL 
      WHEN (mt.excluded_client_list IS NOT NULL 
      AND mt.excluded_client_list != '||' 
      AND mt.excluded_client_list NOT LIKE '%|ispot|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
WHEN 'ispot' != 'nielsen' AND mt.next_nielsen_exclusive THEN NULL 
WHEN False = False AND mt.next_station_vod = True THEN NULL 
WHEN 'ispot' = 'nielsen'   THEN mt.tms_next_channel_callsign 
WHEN 'TMS' = 'TMS'  THEN COALESCE(mt.tms_next_channel_callsign, mt.tivo_next_channel_callsign) 
WHEN 'TMS' = 'TIVO' THEN COALESCE(mt.tivo_next_channel_callsign, mt.tms_next_channel_callsign) 
END as next_channel_callsign, 
CASE WHEN False THEN NULL 
      WHEN (mt.excluded_client_list IS NOT NULL 
      AND mt.excluded_client_list != '||' 
      AND mt.excluded_client_list NOT LIKE '%|ispot|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
WHEN 'ispot' != 'nielsen' AND mt.next_nielsen_exclusive THEN NULL 
WHEN False = False AND mt.next_station_vod = True THEN NULL 
WHEN 'ispot' = 'nielsen'   THEN mt.tms_next_network_affiliate 
WHEN 'TMS' = 'TMS'  THEN COALESCE(mt.tms_next_network_affiliate, mt.tivo_next_network_affiliate) 
WHEN 'TMS' = 'TIVO' THEN COALESCE(mt.tivo_next_network_affiliate, mt.tms_next_network_affiliate) 
END as next_network_affiliate, 
CASE 
WHEN False THEN NULL 
      WHEN (mt.excluded_client_list IS NOT NULL 
      AND mt.excluded_client_list != '||' 
      AND mt.excluded_client_list NOT LIKE '%|ispot|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
WHEN 'ispot' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
WHEN False = False AND mt.prev_station_vod = True THEN NULL 
ELSE mt.live 
END as live, 
mt.brand_name as brand_name, 
mt.title as title, 
mt.duration as duration, 
mt.ip as ip, 
mt.input_category as input_category, 
mt.input_device as input_device, 
CASE WHEN mt.prev_vizio_epg_not_null = False 
THEN CASE WHEN mt.appb_clients <=> '|ALL|' AND '' <=> mt.app_service THEN mt.app_service 
            WHEN mt.appb_clients IS NOT NULL AND mt.appb_clients != '||' AND mt.appb_clients NOT LIKE '%|ispot|%' THEN 'OBFUSCATED' 
            WHEN mt.appb_clients <=> '|ALL|' THEN 'OBFUSCATED' 
            ELSE mt.app_service END 
ELSE mt.app_service 
END as app_service, 
mt.vizio_epg_channel_id as vizio_epg_channel_id, 
mt.vizio_epg_program_id as vizio_epg_program_id
FROM (
      SELECT viz.*
      FROM prod.detection.viewing_commercials_dedup_golden AS viz
      WHERE CASE WHEN (viz.acrb_clients IS NOT NULL
            AND viz.acrb_clients != '||'
            AND viz.acrb_clients NOT LIKE '%|ispot|%') OR viz.acrb_clients <=> '|ALL|' THEN FALSE ELSE TRUE END
      UNION
      SELECT onn.*
      FROM prod.detection.viewing_commercials_dedup_golden AS onn
      WHERE CASE WHEN (onn.acrb_clients IS NOT NULL
            AND onn.acrb_clients != '||'
            AND onn.acrb_clients NOT LIKE '%|ispot|%') OR onn.acrb_clients <=> '|ALL|' THEN FALSE ELSE TRUE END
) AS mt
WHERE (mt.prev_vizio_epg_not_null != False
      OR (NOT (mt.acrb_clients <=> '|ALL|' AND '' <=> mt.app_service))
      OR (NOT (mt.acrb_clients IS NOT NULL AND mt.acrb_clients != '||' AND mt.acrb_clients NOT LIKE '%|ispot|%')
            AND NOT(mt.acrb_clients <=> '|ALL|')
      )
)
AND mt.commercial_client = 'ispot'
and mt.fk_commercial_source_id IN (1)

In [0]:
%sql
CREATE OR REPLACE VIEW dev.mohit_gangwani.test_enhanced_ispot_refined AS
SELECT /*+ BROADCAST(tp)
         */
mt.tvid AS tvid,
mt.zipcode as zipcode, 
mt.dma as dma, 
mt.external_id as value, 
mt.mt_start as mt_start, 
mt.session_start as ts_start, 
mt.session_end as ts_end, 
CASE WHEN mt.prev_filter THEN NULL
     ELSE mt.tms_prev_episode_id
END as prev_episode_id, 
CASE WHEN mt.prev_filter THEN NULL
     ELSE COALESCE(mt.tms_prev_show_title, mt.tivo_prev_show_title)
END as prev_title, 
mt.prev_ts_start as prev_ts_start, 
mt.prev_ts_end as prev_ts_end, 
CASE WHEN mt.prev_filter THEN NULL
     ELSE COALESCE(mt.tms_prev_channel_callsign, mt.tivo_prev_channel_callsign)
END as prev_channel_callsign,
CASE WHEN mt.prev_filter THEN NULL
     ELSE COALESCE(mt.tms_prev_network_affiliate, mt.tivo_prev_network_affiliate)
END as prev_network_affiliate,
CASE WHEN mt.next_filter THEN NULL 
     ELSE mt.tms_next_episode_id
END as next_episode_id, 
CASE WHEN mt.next_filter THEN NULL
     ELSE COALESCE(mt.tms_next_show_title, mt.tivo_next_show_title)
END as next_title, 
mt.next_ts_start as next_ts_start, 
mt.next_ts_end as next_ts_end, 
CASE WHEN mt.next_filter THEN NULL
     ELSE COALESCE(mt.tms_next_channel_callsign, mt.tivo_next_channel_callsign)
END as next_channel_callsign, 
CASE WHEN mt.next_filter THEN NULL 
     ELSE COALESCE(mt.tms_next_network_affiliate, mt.tivo_next_network_affiliate) 
END as next_network_affiliate,
CASE WHEN mt.prev_filter THEN NULL 
     ELSE mt.live 
END as live, 
mt.brand_name as brand_name, 
mt.title as title, 
mt.duration as duration, 
mt.ip as ip, 
mt.input_category as input_category, 
mt.input_device as input_device, 
CASE WHEN mt.prev_vizio_epg_not_null = False 
     THEN CASE WHEN mt.appb_clients <=> '|ALL|' AND '' <=> mt.app_service THEN mt.app_service 
               WHEN mt.appb_clients IS NOT NULL AND mt.appb_clients != '||' AND mt.appb_clients NOT LIKE '%|ispot|%' THEN 'OBFUSCATED' 
               WHEN mt.appb_clients <=> '|ALL|' THEN 'OBFUSCATED' 
               ELSE mt.app_service END
ELSE mt.app_service 
END as app_service, 
mt.vizio_epg_channel_id as vizio_epg_channel_id, 
mt.vizio_epg_program_id as vizio_epg_program_id
FROM (
     SELECT viz.*,
     (COALESCE(viz.excluded_client_list, '||') != '||' AND viz.excluded_client_list NOT LIKE '%|ispot|%')
     OR viz.excluded_client_list <=> '|ALL|'
     OR viz.prev_nielsen_exclusive
     OR viz.prev_station_vod AS prev_filter, 
     (COALESCE(viz.excluded_client_list, '||') != '||' AND viz.excluded_client_list NOT LIKE '%|ispot|%')
     OR viz.excluded_client_list <=> '|ALL|'
     OR viz.next_nielsen_exclusive
     OR viz.next_station_vod AS next_filter
     FROM prod.detection.viewing_commercials_dedup_golden AS viz
     WHERE viz.fk_commercial_source_id = 1
       AND viz.commercial_client = 'ispot'
     UNION ALL
     SELECT onn.*,
     (COALESCE(onn.excluded_client_list, '||') != '||' AND onn.excluded_client_list NOT LIKE '%|ispot|%')
     OR onn.excluded_client_list <=> '|ALL|'
     OR onn.prev_nielsen_exclusive
     OR onn.prev_station_vod AS prev_filter, 
     (COALESCE(onn.excluded_client_list, '||') != '||' AND onn.excluded_client_list NOT LIKE '%|ispot|%')
     OR onn.excluded_client_list <=> '|ALL|'
     OR onn.next_nielsen_exclusive
     OR onn.next_station_vod AS next_filter
     FROM prod.detection.viewing_commercials_dedup_golden AS onn
     WHERE onn.fk_commercial_source_id = 1
       AND onn.commercial_client = 'ispot'
) AS mt
WHERE COALESCE(mt.prev_vizio_epg_not_null, TRUE)
   OR (mt.acrb_clients <=> '|ALL|' AND '' <=> mt.app_service)
   OR COALESCE(mt.acrb_clients, '||') = '||'
   OR mt.acrb_clients LIKE '%|vizio|%'
GROUP BY ALL

## Performance Benchmarks

The benchmark section below uses the same 3-day filter for both views:

```sql
WHERE ts_start >= CURRENT_DATE - INTERVAL 3 DAY
  AND ts_start < CURRENT_DATE
```

Recommended order:

1. Run the two `EXPLAIN COST` cells first to compare the optimized logical plans
2. Run the existing benchmark cell
3. Run the refined benchmark cell
4. Open the query profile for each run and compare wall-clock time, remote bytes, task time, Photon time, and shuffle-heavy operators

Latest benchmark results (both views produce identical output):

| Metric | Existing | Refined |
| --- | --- | --- |
| `COUNT(1)` | 1,309,167,248 | 1,309,167,248 |
| `COUNT(DISTINCT tvid)` | 10,028,320 | 10,028,320 |
| `COUNT(DISTINCT value)` | 231,763 | 231,763 |

Identical counts confirm the refined view preserved the output exactly. The refined view uses `GROUP BY ALL` (equivalent to `SELECT DISTINCT`) — both produce the same deduplicated result from the single-table source.

In [0]:
%sql
-- EXPLAIN: Existing view — look for full scan (no partition filter at scan node)
EXPLAIN COST
SELECT COUNT(1) AS session_count
, COUNT(DISTINCT tvid) AS tv_count
, COUNT(DISTINCT value) AS comm_count
FROM dev.mohit_gangwani.test_enhanced_ispot_existing
WHERE ts_start >= CURRENT_DATE - INTERVAL 3 DAY
AND ts_start < CURRENT_DATE

In [0]:
%sql
-- EXPLAIN: Refined view — look for partition pruning on commercial_client at scan node
EXPLAIN COST
SELECT COUNT(1) AS session_count
, COUNT(DISTINCT tvid) AS tv_count
, COUNT(DISTINCT value) AS comm_count
FROM dev.mohit_gangwani.test_enhanced_ispot_refined
WHERE ts_start >= CURRENT_DATE - INTERVAL 3 DAY
AND ts_start < CURRENT_DATE

## EXPLAIN COST — Plan Comparison & Findings

EXPLAIN COST plans from Cell 12 (existing) and Cell 13 (refined). Benchmark profiles: existing `0cd5355e-7a62-4ba1-973e-99dd337b1c9b`, refined `09a6539f-72f9-4a18-95ce-e88b8b7a0974`.

### What Changed in the Plan

| Area | Existing (Cell 12) | Refined (Cell 13) | Why it matters |
| --- | --- | --- | --- |
| Projection per CASE column | full 4-condition `excluded_client_list` expression inlined in all 9 columns | `CASE WHEN prev_filter#116041 THEN null` / `CASE WHEN next_filter#116042 THEN null` — single attribute reference | eliminates repeated `NOT LIKE '%\|ispot\|%'` string-match per row per column |
| Scan-level filter | `commercial_client` and `fk_commercial_source_id` pushed from outer WHERE indirectly | all conditions pushed directly: `fk_commercial_source_id = 1 AND commercial_client = 'ispot' AND NOT (acrb_clients <=> '\|ALL\|') AND (Contains(acrb_clients, '\|ispot\|') OR coalesce(acrb_clients, '\|\|') = '\|\|')` | explicit co-location with the scan; no indirect predicate path |
| `SELECT DISTINCT` / dedup | `SELECT DISTINCT` on 29 output columns | `GROUP BY ALL` — semantically identical | both produce identical row counts; existing inline expressions cause larger shuffle data movement before the dedup |
| Join to hashes table | join to `prod.customer_reports.hashes_production_vizio` with `/*+ BROADCAST(v) */` hint (silently ignored — 190M rows) | removed — view outputs `mt.tvid` directly | eliminates the join, associated shuffle, and hash table read entirely |
| Outer WHERE | 3-part OR CASE block evaluated post-scan | no outer WHERE — subquery filter is the only access-control gate | removes a no-op evaluation layer from every output row |

### Key Plan Details

**Existing (Cell 12) — projection node:**

Each of the nine prev/next CASE columns in the optimized plan inlines the full Catalyst-folded condition:

```
CASE WHEN (excluded_client_list <=> '|ALL|'
           OR (isnotnull(excluded_client_list)
               AND NOT (excluded_client_list = '||')
               AND NOT Contains(excluded_client_list, '|ispot|'))) THEN null
     WHEN prev_nielsen_exclusive THEN null
     WHEN prev_station_vod THEN null
     WHEN true THEN tms_prev_episode_id
END
```

**Refined (Cell 13) — projection node:**

Each column reduces to a two-branch lookup:

```
CASE WHEN prev_filter#116041 THEN null ELSE tms_prev_episode_id END
```

Catalyst exposes `prev_filter` and `next_filter` as named boolean attributes (computed once in the subquery), visible as `prev_filter#116041` and `next_filter#116042` throughout the plan.

**Refined (Cell 13) — scan filter node:**

All access-control and partition conditions are pushed directly to the relation:

```
Filter (fk_commercial_source_id = 1
    AND commercial_client = 'ispot'
    AND NOT (acrb_clients <=> '|ALL|')
    AND (Contains(acrb_clients, '|ispot|') OR coalesce(acrb_clients, '||') = '||'))
```

**Base table statistics:** `prod.detection.viewing_commercials_dedup_golden` — 3.0 PiB / 2.80E+12 rows raw.

### Measured Benchmark Results

| Metric | Existing | Refined | Delta |
| --- | --- | --- | --- |
| `COUNT(1)` | 1,309,167,248 | 1,309,167,248 | identical |
| `COUNT(DISTINCT tvid)` | 10,028,320 | 10,028,320 | identical |
| `COUNT(DISTINCT value)` | 231,763 | 231,763 | identical |
| Duration | 371.2s (6m 11s) | **93.7s (1m 33.7s)** | **−74.8% (3.96× faster)** |
| Shuffle | 3.44h | 1.78h | **−48.3%** — dominant driver |
| Grouping Aggregate | 42.57m | 1.12h | existing: shuffle-dominated; refined: balanced |
| Task time | 4.98h | 3.68h | **−26.1%** |
| Files read | 124 | 124 | same — single ispot partition |
| Partitions read | 1 | 1 | same pruning |
| Pruned bytes | 251.0 TiB | 251.0 TiB | same pruning |

> Cache note: existing ran first (0% cache hit — all remote). Refined ran second (37% cache hit). The 3.96× improvement holds structurally: existing performs significantly more task work regardless of cache.

### On SELECT DISTINCT / GROUP BY ALL

The existing view uses `SELECT DISTINCT` on 29 output columns; the refined view uses `GROUP BY ALL` (semantically identical — all non-aggregate columns become the group key). Both produce identical row counts, confirming the deduplication produces the same result from the single-table pre-deduplicated source.

The residual dedup overhead (Shuffle + Grouping Aggregate) remains in both plans but is substantially reduced in the refined view due to simpler expression shapes — the Shuffle no longer moves 9 columns of complex CASE-evaluated data (3.44h → 1.78h, −48%). Removing `GROUP BY ALL` from the refined view entirely is a potential further optimization: if the `_dedup_golden` source is guaranteed to contain no duplicate rows for the ispot partition, the dedup is redundant. That would require explicit validation before the change is applied.

In [0]:
%sql
-- Benchmark: Existing view
-- After this runs, open the Query Profile and note: files read, bytes read, duration
SELECT COUNT(1) AS session_count
, COUNT(DISTINCT tvid) AS tv_count
, COUNT(DISTINCT value) AS comm_count
FROM dev.mohit_gangwani.test_enhanced_ispot_existing
WHERE ts_start >= CURRENT_DATE - INTERVAL 3 DAY
AND ts_start < CURRENT_DATE

In [0]:
%sql
-- Benchmark: Refined view
-- After this runs, open the Query Profile and compare files read, bytes read, duration vs. existing view above
SELECT COUNT(1) AS session_count
, COUNT(DISTINCT tvid) AS tv_count
, COUNT(DISTINCT value) AS comm_count
FROM dev.mohit_gangwani.test_enhanced_ispot_refined
WHERE ts_start >= CURRENT_DATE - INTERVAL 3 DAY
AND ts_start < CURRENT_DATE

In [0]:
 %sql
-- Benchmark: Refined view
-- After this runs, open the Query Profile and compare files read, bytes read, duration vs. existing view above
SELECT COUNT(1) AS session_count
, COUNT(DISTINCT tvid) AS tv_count
, COUNT(DISTINCT value) AS comm_count
FROM prod.datalake_share.enhanced_ispot_commercial
WHERE ts_start >= CURRENT_DATE - INTERVAL 3 DAY
AND ts_start < CURRENT_DATE

In [0]:
%sql
ALTER VIEW cure_deltashare.vizio.enhanced_ispot_commercial AS
SELECT 
mt.tvid AS tvid,
mt.zipcode as zipcode, 
mt.dma as dma, 
mt.external_id as value, 
mt.mt_start as mt_start, 
mt.session_start as ts_start, 
mt.session_end as ts_end, 
CASE WHEN mt.prev_filter THEN NULL
     ELSE mt.tms_prev_episode_id
END as prev_episode_id, 
CASE WHEN mt.prev_filter THEN NULL
     ELSE COALESCE(mt.tms_prev_show_title, mt.tivo_prev_show_title)
END as prev_title, 
mt.prev_ts_start as prev_ts_start, 
mt.prev_ts_end as prev_ts_end, 
CASE WHEN mt.prev_filter THEN NULL
     ELSE COALESCE(mt.tms_prev_channel_callsign, mt.tivo_prev_channel_callsign)
END as prev_channel_callsign,
CASE WHEN mt.prev_filter THEN NULL
     ELSE COALESCE(mt.tms_prev_network_affiliate, mt.tivo_prev_network_affiliate)
END as prev_network_affiliate,
CASE WHEN mt.next_filter THEN NULL 
     ELSE mt.tms_next_episode_id
END as next_episode_id, 
CASE WHEN mt.next_filter THEN NULL
     ELSE COALESCE(mt.tms_next_show_title, mt.tivo_next_show_title)
END as next_title, 
mt.next_ts_start as next_ts_start, 
mt.next_ts_end as next_ts_end, 
CASE WHEN mt.next_filter THEN NULL
     ELSE COALESCE(mt.tms_next_channel_callsign, mt.tivo_next_channel_callsign)
END as next_channel_callsign, 
CASE WHEN mt.next_filter THEN NULL 
     ELSE COALESCE(mt.tms_next_network_affiliate, mt.tivo_next_network_affiliate) 
END as next_network_affiliate,
CASE WHEN mt.prev_filter THEN NULL 
     ELSE mt.live 
END as live, 
mt.brand_name as brand_name, 
mt.title as title, 
mt.duration as duration, 
mt.ip as ip, 
mt.input_category as input_category, 
mt.input_device as input_device, 
CASE WHEN mt.prev_vizio_epg_not_null = False 
     THEN CASE WHEN mt.appb_clients <=> '|ALL|' AND '' <=> mt.app_service THEN mt.app_service 
               WHEN mt.appb_clients IS NOT NULL AND mt.appb_clients != '||' AND mt.appb_clients NOT LIKE '%|vizio|%' THEN 'OBFUSCATED' 
               WHEN mt.appb_clients <=> '|ALL|' THEN 'OBFUSCATED' 
               ELSE mt.app_service END
ELSE mt.app_service 
END as app_service, 
mt.vizio_epg_channel_id as vizio_epg_channel_id, 
mt.vizio_epg_program_id as vizio_epg_program_id,
mt.session_start_hour AS date_partition
FROM (
       SELECT viz.*,
       (COALESCE(viz.excluded_client_list, '||') != '||' AND viz.excluded_client_list NOT LIKE '%|ispot|%')
       OR viz.excluded_client_list <=> '|ALL|'
       OR viz.prev_nielsen_exclusive
       OR viz.prev_station_vod AS prev_filter, 
       (COALESCE(viz.excluded_client_list, '||') != '||' AND viz.excluded_client_list NOT LIKE '%|ispot|%')
       OR viz.excluded_client_list <=> '|ALL|'
       OR viz.next_nielsen_exclusive
       OR viz.next_station_vod AS next_filter
       FROM prod.detection.viewing_commercials_dedup_golden AS viz
       WHERE viz.fk_commercial_source_id = 1
         AND viz.commercial_client = 'ispot'
     UNION ALL
     SELECT onn.*,
       (COALESCE(onn.excluded_client_list, '||') != '||' AND onn.excluded_client_list NOT LIKE '%|ispot|%')
       OR onn.excluded_client_list <=> '|ALL|'
       OR onn.prev_nielsen_exclusive
       OR onn.prev_station_vod AS prev_filter, 
       (COALESCE(onn.excluded_client_list, '||') != '||' AND onn.excluded_client_list NOT LIKE '%|ispot|%')
       OR onn.excluded_client_list <=> '|ALL|'
       OR onn.next_nielsen_exclusive
       OR onn.next_station_vod AS next_filter
       FROM prod.detection_onn.viewing_commercials_dedup_golden AS onn
       WHERE onn.fk_commercial_source_id = 1
         AND onn.commercial_client = 'ispot'
) AS mt
WHERE COALESCE(mt.prev_vizio_epg_not_null, TRUE)
   OR (mt.acrb_clients <=> '|ALL|' AND '' <=> mt.app_service)
   OR COALESCE(mt.acrb_clients, '||') = '||'
   OR mt.acrb_clients LIKE '%|vizio|%'
GROUP BY ALL